# <font color="steelblue">17. Cambio de base y enlace de series</font>

*Unidad 3. Números índice y series cronológicas* · Sesión teórica 17 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`ipc_simulado.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["ipc_simulado.csv"]
PREGUNTAS = ["q_P17_1.json", "q_P17_2.json", "q_P17_3.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Cada cierto tiempo los institutos de estadística **cambian la base** de sus índices y las series antiguas dejan de enlazar con las nuevas. Si quieres contar cómo han evolucionado los precios desde 2015, o comparar dos índices con bases distintas, tendrás que hacer un **cambio de base** o un **enlace**. Son dos operaciones sencillas que evitan errores muy visibles.

**Qué necesitas saber antes**: los cuadernos 14 y 16.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Cambiar la base de un índice a otro periodo.
2. Enlazar dos tramos de una serie publicados con bases distintas.
3. Calcular la variación entre dos fechas directamente con el índice.

## <font color='steelblue'>Contenidos</font>

1. Cambiar la base
2. Variación entre dos fechas
3. Enlazar dos tramos de una serie

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Cambiar la base</font>

In [ ]:
ipc = pd.read_csv("ipc_simulado.csv")
ipc["fecha"] = pd.to_datetime(ipc["fecha"])
ipc = ipc.set_index("fecha")

Para pasar un índice a una nueva base, se divide toda la serie por el valor (o la media) del nuevo periodo base y se multiplica por 100:

$$I^{nueva}_t = \frac{I_t}{I_{nueva\ base}} \times 100$$

Las **tasas de variación no cambian**: solo cambia la referencia.

In [ ]:
base_2015 = ipc["general"]["2015"].mean()
ipc["general_b2015"] = (ipc["general"] / base_2015 * 100).round(2)
print(ipc[["general", "general_b2015"]].iloc[[0, 72, -1]])

In [ ]:
tasa_b2021 = ipc["general"].pct_change(12).iloc[-1] * 100
tasa_b2015 = ipc["general_b2015"].pct_change(12).iloc[-1] * 100
print(f"Inflación con base 2021: {tasa_b2021:.2f} %   con base 2015: {tasa_b2015:.2f} %")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P17_1.json")

## <font color='steelblue'>2. Variación entre dos fechas</font>

Con un índice se calcula directamente cuánto han subido los precios entre dos fechas cualesquiera, sin importar la base:

In [ ]:
def variacion(serie, desde, hasta):
    return (serie[hasta] / serie[desde] - 1) * 100

print(f"Precios enero 2015 - diciembre 2025: +{variacion(ipc['general'], '2015-01-01', '2025-12-01'):.1f} %")
print(f"Lo mismo con base 2015:           +{variacion(ipc['general_b2015'], '2015-01-01', '2025-12-01'):.1f} %")

⚠️ Un error frecuente es **restar** los índices: pasar de 105 a 115 no es «un 10 % más», sino (115 / 105 − 1) × 100 = 9,5 %. Restar solo da el porcentaje correcto si el punto de partida vale 100.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P17_2.json")

## <font color='steelblue'>3. Enlazar dos tramos de una serie</font>

Imagina que un organismo publicó la serie hasta 2020 con base 2016 y desde 2021 con base 2021. Para tener una sola serie se busca un **periodo de solapamiento** (un mes que esté en las dos) y se calcula un **coeficiente de enlace**.

In [ ]:
general = ipc["general"]
antigua = (general[:"2021-01-01"] / general["2016"].mean() * 100).round(2)     # publicada en base 2016
nueva = general["2021-01-01":].round(3)                                      # publicada en base 2021
print("Mes común (enero 2021):  antigua =", antigua["2021-01-01"], "  nueva =", nueva["2021-01-01"])

In [ ]:
coef = nueva["2021-01-01"] / antigua["2021-01-01"]
antigua_enlazada = antigua * coef
serie_enlazada = pd.concat([antigua_enlazada[:"2020-12-01"], nueva])
print("Coeficiente de enlace:", round(coef, 4))
serie_enlazada.plot(color="steelblue")
plt.title("Serie enlazada en base 2021 (IPC simulado)")
plt.show()

**Qué ha pasado**: hemos reescalado el tramo antiguo para que coincida con el nuevo en el mes común. Los institutos oficiales suelen publicar ya las series enlazadas; cuando lo hagan, úsalas en lugar de fabricarlas tú.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P17_3.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Pon el índice de alimentos en base enero 2020 = 100.

<details><summary>Ver solución</summary>

```python
(ipc["alimentos"] / ipc.loc["2020-01-01", "alimentos"] * 100).round(1)
```
</details>

### **Ejercicio 2** *(sección 2)*

¿Cuánto han subido los alimentos y la vivienda entre enero de 2021 y diciembre de 2025?

<details><summary>Ver solución</summary>

```python
for g in ["alimentos", "vivienda"]:
    print(g, round(variacion(ipc[g], "2021-01-01", "2025-12-01"), 1))
```
</details>

### **Ejercicio 3** *(sección 3)*

Enlaza los tramos usando como mes común junio de 2021 en lugar de enero. ¿Cambia el resultado?

<details><summary>Ver solución</summary>

```python
a = general[:"2021-06-01"] / general["2016"].mean() * 100
n = general["2021-06-01":]
(a * n["2021-06-01"] / a["2021-06-01"]).tail(3)
```

No cambia, porque los dos tramos salen aquí de la misma serie. Con series oficiales reales puede haber pequeñas diferencias según el mes de enlace.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- **Cambio de base**: dividir la serie por el valor del nuevo periodo base y multiplicar por 100; las tasas no cambian.
- La variación entre dos fechas es **cociente − 1**, no la resta de índices (salvo que el inicio valga 100).
- Para **enlazar** dos tramos con bases distintas se usa un periodo común y un coeficiente de enlace.
- Si el organismo publica series enlazadas oficiales, úsalas.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Periodo base** | Periodo cuyo índice vale 100. |
| **Cambio de base** | Reexpresar un índice respecto a otro periodo base. |
| **Enlace** | Unión de dos tramos de una serie publicados con bases distintas. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 18 empieza el análisis de series cronológicas: componentes y gráficos.

**Siguiente cuaderno**: [18. Series cronológicas: componentes y lectura de gráficos temporales](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P18_series_cronologicas.ipynb)